# 00: Benchmark Summary

*How do all evaluated KGs compare across the full BioKGSuite benchmark?*

This notebook aggregates results from the seven dimension notebooks (01-07) and produces summary visualisations and tables for reporting.

| Output | Description |
|--------|-------------|
| **Radar chart** | All 7 dimensions as a polar quality profile per KG |
| **Category bar chart** | Mean score per category group (Content / Structure / Inference) |
| **18-metric heatmap** | Sub-metric detail for all 18 metrics across all dimensions |
| **Parallel coordinates** | Sub-metric score profiles threading across all 18 metrics |
| **Lollipop chart** | Dimension-level scores and inter-KG variability |
| **Summary table** | Full numeric table exported to CSV |

**Inputs:** `results/checkpoints/01_coverage.pkl` through `07_generalization.pkl`

**Outputs:** `results/figures/00_benchmark_radar.{pdf,png}` - `00_benchmark_categories.{pdf,png}` - `00_benchmark_heatmap.{pdf,png}` - `00_parallel_coordinates.{pdf,png}` - `00_benchmark_lollipop.{pdf,png}` - `results/tables/00_benchmark_summary.csv`

**Dependencies:** `src/loading.py` - `src/plotting.py`

## Set-up

Checkpoints are read from `results/checkpoints/`; any missing file is excluded from all aggregates with a printed warning rather than substituting a default value. KG names are read from `config.yaml` - add or remove a KG block there without modifying notebook code.

In [1]:
# Imports
import sys, os, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from typing import Optional, List, Dict

_root = Path(os.path.abspath('')).resolve()
_root = _root.parent if _root.name == 'eval_notebooks' else _root
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import math
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
from src.plotting import (setup_style, save_fig, clean_ax, panel_label,
                           TEXT_COLOR, TICK_COLOR, ALERT_RED, KG_PALETTE,
                           DOUBLE_COL_W, ROW_H_STD)
from src.loading  import find_config, load_config

setup_style()

In [2]:
# Config and paths
config    = load_config(find_config(_root))
BASE      = config['_base_dir']
FIGS      = BASE / 'results' / 'figures'
CKPT_DIR  = BASE / 'results' / 'checkpoints'

KG_NAMES  = list(config['knowledge_graphs'].keys())
KG_COLORS = {name: KG_PALETTE.get(name, '#888888') for name in KG_NAMES}

FIGS.mkdir(parents=True, exist_ok=True)

def _style_left_spine(ax):
    # Hide top/right/bottom spines; style left spine for clean axis presentation.
    for sp in ('top', 'right', 'bottom'):
        ax.spines[sp].set_visible(False)
    ax.spines['left'].set_color('#333333')
    ax.spines['left'].set_linewidth(0.5)

## 1. Checkpoint loading

In [3]:
# Checkpoint loading - missing files are excluded from all aggregates
CKPT_FILES: Dict[str, str] = {
    'coverage':            '01_coverage.pkl',
    'annotation_accuracy': '02_semantic_validity.pkl',
    'trustworthiness':     '03_trustworthiness.pkl',
    'topology':            '04_topology.pkl',
    'stability':           '05_stability.pkl',
    'task_performance':    '06_predictive_performance.pkl',
    'generalisation':      '07_generalization.pkl',
}

# Map checkpoint files back to their source notebooks for freshness check
_CKPT_SOURCE_NB: Dict[str, str] = {
    '01_coverage.pkl':               '01_coverage.ipynb',
    '02_semantic_validity.pkl':      '02_annotation_accuracy.ipynb',
    '03_trustworthiness.pkl':        '03_trustworthiness.ipynb',
    '04_topology.pkl':               '04_topology.ipynb',
    '05_stability.pkl':              '05_stability.ipynb',
    '06_predictive_performance.pkl': '06_task_performance.ipynb',
    '07_generalization.pkl':         '07_generalization.ipynb',
}

_NB_DIR = BASE / 'eval_notebooks'

ckpts: Dict[str, dict] = {}

for _dim, _fname in CKPT_FILES.items():
    _path = CKPT_DIR / _fname
    if _path.exists():
        with open(_path, 'rb') as f:
            ckpts[_dim] = pickle.load(f)
    else:
        print(f'  MISSING {_dim} ({_fname}) - excluded from aggregates')

LOADED_DIMS = list(ckpts.keys())
DIMS        = list(CKPT_FILES.keys())
_loaded_str = ', '.join(LOADED_DIMS)
print(f'\n{len(LOADED_DIMS)}/{len(CKPT_FILES)} dimensions loaded: {_loaded_str}')



7/7 dimensions loaded: coverage, annotation_accuracy, trustworthiness, topology, stability, task_performance, generalisation


## 2. Score matrix

In [4]:
# Build unified score matrix
CATEGORIES: Dict[str, List[str]] = {
    'Content':   ['coverage', 'annotation_accuracy', 'trustworthiness'],
    'Structure': ['topology', 'stability'],
    'Inference': ['task_performance', 'generalisation'],
}

DIM_LABELS: Dict[str, str] = {
    'coverage':            'Coverage',
    'annotation_accuracy': 'Annotation Accuracy',
    'trustworthiness':     'Trustworthiness',
    'topology':            'Topology',
    'stability':           'Stability',
    'task_performance':    'Task Performance',
    'generalisation':      'Generalisation',
}

# summary_scores[kg][dim] = dimension-level float score (0-1)
summary_scores: Dict[str, Dict[str, float]] = {kg: {} for kg in KG_NAMES}
for _dim, _ck in ckpts.items():
    for _kg in KG_NAMES:
        _v = _ck['summary_scores'].get(_kg)
        if _v is not None:
            summary_scores[_kg][_dim] = float(_v)

# category_scores[kg][cat] = mean of loaded dims in that category
category_scores: Dict[str, Dict[str, Optional[float]]] = {}
for _kg in KG_NAMES:
    category_scores[_kg] = {}
    for _cat, _cat_dims in CATEGORIES.items():
        _vals = [summary_scores[_kg][_d] for _d in _cat_dims if _d in summary_scores[_kg]]
        category_scores[_kg][_cat] = round(float(np.mean(_vals)), 4) if _vals else None

# Print dimension scores table
_cw = 11
print('Dimension scores (0-1, higher = better)')
print(f'  {"KG":<12s}', end='')
for _d in DIMS:
    print(f'  {_d[:_cw]:>{_cw}s}', end='')
print()
print('  ' + '-' * (14 + (_cw + 2) * len(DIMS)))
for _kg in KG_NAMES:
    print(f'  {_kg:<12s}', end='')
    for _d in DIMS:
        _v = summary_scores[_kg].get(_d)
        print(f'  {_v:>{_cw}.3f}' if _v is not None else f'  {"N/A":>{_cw}s}', end='')
    print()

# Print category scores table
print()
print('Category scores (mean across dimensions in each category)')
print(f'  {"KG":<12s}', end='')
for _cat in CATEGORIES:
    print(f'  {_cat:>12s}', end='')
print()
print('  ' + '-' * (14 + 14 * len(CATEGORIES)))
for _kg in KG_NAMES:
    print(f'  {_kg:<12s}', end='')
    for _cat in CATEGORIES:
        _v = category_scores[_kg].get(_cat)
        print(f'  {_v:>12.3f}' if _v is not None else f'  {"N/A":>12s}', end='')
    print()

Dimension scores (0-1, higher = better)
  KG               coverage  annotation_  trustworthi     topology    stability  task_perfor  generalisat
  ---------------------------------------------------------------------------------------------------------
  primekg             0.583        1.000        0.387        0.775        0.979        0.581        0.823
  hetionet            0.266        1.000        0.247        0.659        0.990        0.402        0.839
  drkg                0.463        0.999        0.476        0.723        0.963        0.516        0.737
  openbilink          0.421        1.000        0.509        0.690        0.894        0.549        0.740
  biokg               0.465        1.000        0.188        0.707        0.999        0.378        0.734
  matrix              0.775        1.000        0.978        0.766        1.000        0.500        0.960

Category scores (mean across dimensions in each category)
  KG                 Content     Structure     Infe

## 3. Radar chart (all 7 dimensions)

In [5]:
# Radar/spider chart - single plot with all 7 dimensions
_CAT_SHORT: Dict[str, str] = {
    'coverage':            'Coverage',
    'annotation_accuracy': 'Annotation\nAccuracy',
    'trustworthiness':     'Trustworthiness',
    'topology':            'Topology',
    'stability':           'Stability',
    'task_performance':    'Task\nPerformance',
    'generalisation':      'Generalisation',
}

_avail_dims = [d for d in DIMS if d in LOADED_DIMS]
_labels     = [_CAT_SHORT.get(d, DIM_LABELS.get(d, d)) for d in _avail_dims]
N           = len(_avail_dims)
angles      = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles     += angles[:1]

fig, ax = plt.subplots(figsize=(6.0, 6.0), subplot_kw={'polar': True})

ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(_labels, fontsize=11, color=TEXT_COLOR)
# Push tick labels outward so they don't overlap the polygon
ax.tick_params(axis='x', pad=14)
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(['0.25', '0.50', '0.75', '1.00'], fontsize=9, color=TICK_COLOR)
ax.spines['polar'].set_visible(False)
ax.grid(color='#E0E0E0', lw=0.5)

for _kg in KG_NAMES:
    _vals   = [summary_scores[_kg].get(d, 0.0) for d in _avail_dims]
    _closed = _vals + _vals[:1]
    ax.plot(angles, _closed, color=KG_COLORS[_kg], lw=2.0, label=_kg.upper())
    ax.fill(angles, _closed, color=KG_COLORS[_kg], alpha=0.08)

ax.legend(fontsize=10, frameon=False, bbox_to_anchor=(0.5, -0.10),
          loc='upper center', ncol=len(KG_NAMES))
ax.set_title('Multi-dimensional quality profile of biomedical KGs',
             fontsize=11, fontweight='bold', color=TEXT_COLOR, pad=24)

plt.tight_layout()
save_fig(fig, FIGS, '00_benchmark_radar')
plt.show()

  → Saved: 00_benchmark_radar.pdf / .png


## 4. Category scores

In [6]:
# Category bar chart - grouped bars, x-axis = categories, bars = KGs
_cats = list(CATEGORIES.keys())
_x    = np.arange(len(_cats))
_w    = 0.8 / max(len(KG_NAMES), 1)

fig, ax = plt.subplots(figsize=(DOUBLE_COL_W * 0.65, 3.1))

for _i, _kg in enumerate(KG_NAMES):
    _vals = [category_scores[_kg].get(_c) for _c in _cats]
    _xs   = _x + (_i - (len(KG_NAMES) - 1) / 2) * _w
    _bars = ax.bar(_xs, [_v if _v is not None else 0 for _v in _vals], _w * 0.9,
                   color=KG_COLORS[_kg], label=_kg.upper(),
                   edgecolor='white', linewidth=0.5)
    for _bar, _v in zip(_bars, _vals):
        if _v is not None:
            ax.text(_bar.get_x() + _bar.get_width() / 2, _bar.get_height() + 0.015,
                    f'{_v:.2f}', ha='center', va='bottom', rotation=90,
                    fontsize=7, color=TEXT_COLOR)

ax.axhline(0.5, color='#868074', ls='--', lw=0.9, alpha=0.7, zorder=0)
ax.axhline(0.8, color='#655F55', ls='--', lw=0.9, alpha=0.7, zorder=0)
ax.set_xticks(_x)
ax.set_xticklabels(_cats, fontsize=9.5, color=TICK_COLOR)
ax.set_ylim(0, 1.18)
ax.set_yticks(np.arange(0, 1.01, 0.2))
ax.set_ylabel('Mean score', fontsize=9, color=TICK_COLOR)
ax.margins(x=0.02)

ax.legend(fontsize=7.5, frameon=False, ncol=3, loc='upper center',
          bbox_to_anchor=(0.5, -0.13), columnspacing=1.4,
          handlelength=1.2, handletextpad=0.5)

_style_left_spine(ax)
clean_ax(ax, title='Aggregate category scores across evaluation dimensions')
plt.tight_layout()
save_fig(fig, FIGS, '00_benchmark_categories')
plt.show()

  → Saved: 00_benchmark_categories.pdf / .png


In [7]:
# Full 18-metric heatmap - sub-metric extraction helpers
from src.plotting import HEATMAP_CMAP
from matplotlib.patches import Rectangle
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

def _safe_mean(vals):
    # Mean of non-None, non-NaN values; returns None if list is empty.
    _clean = [_v for _v in vals
              if _v is not None and not (isinstance(_v, float) and math.isnan(_v))]
    return float(np.mean(_clean)) if _clean else None

def _stab_score(r):
    # Map Spearman r in (-1, 1) to [0, 1]; 0.5 = no correlation.
    if r is None or (isinstance(r, float) and math.isnan(r)):
        return None
    return round((float(r) + 1) / 2, 4)

def _x_stability(ck, kg, strategy):
    _rates = ck.get('dropout_rates', [])
    if not _rates:
        return None
    _min_r = min(_rates)
    _r = (ck.get('stab_scalars', {})
            .get(kg, {})
            .get(_min_r, {})
            .get(strategy, {})
            .get('spearman_r'))
    return _stab_score(_r)

def _x_nbhd_retrieval(ck, kg):
    return _safe_mean([
        _r.get('auroc') for _r in ck.get('link_records', [])
        if _r.get('kg') == kg
        and _r.get('strategy') == 'shared-target'
        and _r.get('auroc') is not None
    ])

def _x_moa(ck, kg):
    return ck.get('moa_metrics', {}).get(kg, {}).get('target_on_path_rate')

# 17 sub-metric definitions (column_label, dim_key, extractor_fn(ck, kg))
SUB_METRICS = [
    # Coverage (2)
    ('Entity Coverage',       'coverage',            lambda ck, kg: ck.get('sub_scores', {}).get(kg, {}).get('entity_coverage')),
    ('Relation Coverage',     'coverage',            lambda ck, kg: ck.get('sub_scores', {}).get(kg, {}).get('relation_coverage')),
    # Annotation Accuracy (2)
    ('Entity Validity',   'annotation_accuracy', lambda ck, kg: ck.get('sub_scores', {}).get(kg, {}).get('entity_validity')),
    ('Relational Consistency',  'annotation_accuracy', lambda ck, kg: ck.get('sub_scores', {}).get(kg, {}).get('relational_consistency')),
    # Trustworthiness (2 scored; source diversity is descriptive only)
    ('Edge Traceability',     'trustworthiness',     lambda ck, kg: ck.get('sub_scores', {}).get(kg, {}).get('edge_traceability')),
    ('Uncertainty Quantification',       'trustworthiness',     lambda ck, kg: ck.get('sub_scores', {}).get(kg, {}).get('uq_coverage')),
    # Topology (4)
    ('Connectedness',     'topology',            lambda ck, kg: ck.get('sub_scores', {}).get(kg, {}).get('connectedness')),
    ('Small-World',       'topology',            lambda ck, kg: ck.get('sub_scores', {}).get(kg, {}).get('small_world')),
    ('Reachability',      'topology',            lambda ck, kg: ck.get('sub_scores', {}).get(kg, {}).get('reachability')),
    ('Community Purity',  'topology',            lambda ck, kg: ck.get('sub_scores', {}).get(kg, {}).get('community_purity')),
    # Stability (2)
    ('Random Dropout',    'stability',           lambda ck, kg: ck.get('sub_scores', {}).get(kg, {}).get('random_stability')),
    ('Periph. Dropout',   'stability',           lambda ck, kg: ck.get('sub_scores', {}).get(kg, {}).get('periphery_stability')),
    # Task Performance (3)
    ('Link Prediction',        'task_performance',    lambda ck, kg: ck.get('sub_scores', {}).get(kg, {}).get('link_prediction')),
    ('Neighborhood Retrieval',   'task_performance',    lambda ck, kg: ck.get('sub_scores', {}).get(kg, {}).get('nbhd_retrieval')),
    ('Multi-hop Reasoning',   'task_performance',    lambda ck, kg: ck.get('sub_scores', {}).get(kg, {}).get('multihop_reasoning')),
    # Generalisation (3)
    ('Data-Sparse Gen.',  'generalisation',      lambda ck, kg: ck.get('sub_scores', {}).get(kg, {}).get('data_sparse_gen')),
    ('Cross-Domain Gen.', 'generalisation',      lambda ck, kg: ck.get('sub_scores', {}).get(kg, {}).get('cross_domain_gen')),
    ('Prospective Gen.',     'generalisation',      lambda ck, kg: ck.get('sub_scores', {}).get(kg, {}).get('temporal_gen')),
]

# Build heatmap matrix (rows=KGs, cols=sub-metrics)
heat_data = np.full((len(KG_NAMES), len(SUB_METRICS)), np.nan)
for _j, (_, _dim, _extractor) in enumerate(SUB_METRICS):
    _ck = ckpts.get(_dim)
    if _ck is None:
        continue
    for _i, _kg in enumerate(KG_NAMES):
        _v = _extractor(_ck, _kg)
        if _v is not None and not (isinstance(_v, float) and math.isnan(_v)):
            heat_data[_i, _j] = float(_v)

# Dimension group boundaries (column indices where a new dim starts)
_dim_boundaries, _prev_dim = [], None
for _j, (_, _dim, _) in enumerate(SUB_METRICS):
    if _dim != _prev_dim:
        _dim_boundaries.append(_j)
        _prev_dim = _dim

# Draw heatmap using Rectangle patches (no anti-aliasing seams)
_N_cols = len(SUB_METRICS)
_N_rows = len(KG_NAMES)
_fig_w  = max(DOUBLE_COL_W, _N_cols * 0.54 + 1.6)
_fig_h  = _N_rows * 0.58 + 3.0
fig, ax = plt.subplots(figsize=(_fig_w, _fig_h))

_norm = Normalize(vmin=0, vmax=1)
_na_color = '#EDEADF'

for _i in range(_N_rows):
    for _j in range(_N_cols):
        _v = heat_data[_i, _j]
        if np.isnan(_v):
            _fc = _na_color
        else:
            _fc = HEATMAP_CMAP(_norm(_v))
        _rect = Rectangle((_j - 0.5, _i - 0.5), 1, 1,
                           facecolor=_fc, edgecolor=_fc, linewidth=0.5)
        ax.add_patch(_rect)

        # Cell text
        if not np.isnan(_v):
            _col = 'white' if _v >= 0.65 else TEXT_COLOR
            ax.text(_j, _i, f'{_v:.2f}', ha='center', va='center',
                    fontsize=7.5, fontweight='bold', color=_col)
        else:
            ax.text(_j, _i, 'N/A', ha='center', va='center',
                    fontsize=6.5, color='#A8A296')

ax.set_xlim(-0.5, _N_cols - 0.5)
ax.set_ylim(_N_rows - 0.5, -0.5)
ax.set_aspect('equal')

# Dimension group labels above columns
_dim_groups = []
_prev_dim, _gstart = None, 0
for _j, (_, _dim, _) in enumerate(SUB_METRICS):
    if _dim != _prev_dim and _prev_dim is not None:
        _dim_groups.append((_prev_dim, _gstart, _j - 1))
        _gstart = _j
    _prev_dim = _dim
_dim_groups.append((_prev_dim, _gstart, _N_cols - 1))

for _dim_key, _jstart, _jend in _dim_groups:
    _mid_x = (_jstart + _jend) / 2.0
    ax.text(_mid_x, -0.75, DIM_LABELS.get(_dim_key, _dim_key),
            ha='center', va='bottom', fontsize=7.5,
            fontweight='bold', color=TEXT_COLOR,
            clip_on=False)

ax.set_xticks(range(_N_cols))
ax.set_xticklabels([_m[0] for _m in SUB_METRICS], rotation=45, ha='right',
                   fontsize=8, color=TICK_COLOR)
ax.set_yticks(range(_N_rows))
ax.set_yticklabels([_kg.upper() for _kg in KG_NAMES], fontsize=9, color=TICK_COLOR)
ax.tick_params(length=0)
for _sp in ax.spines.values():
    _sp.set_visible(False)

# Colorbar from a ScalarMappable (no imshow image to reference)
_sm = ScalarMappable(cmap=HEATMAP_CMAP, norm=_norm)
_sm.set_array([])
plt.colorbar(_sm, ax=ax, shrink=0.55, pad=0.01, label='Score (0-1)')

ax.set_title('Metric performance across knowledge graphs',
             fontsize=11, fontweight='bold', color=TEXT_COLOR, pad=42)
plt.tight_layout()
save_fig(fig, FIGS, '00_benchmark_heatmap')
plt.show()

  → Saved: 00_benchmark_heatmap.pdf / .png


## 5. Summary table

In [8]:
# Summary table for paper - KGs x (18 sub-metrics + 7 dim scores + overall mean)
_DIM_SCORE_SUFFIX = ' [dim]'
_OVERALL_COL      = 'Overall Mean'

# Column list sub-metrics, dim scores, overall
_sub_cols = [_m[0] for _m in SUB_METRICS]
_dim_cols = [DIM_LABELS.get(_d, _d) + _DIM_SCORE_SUFFIX for _d in DIMS]
_all_cols = _sub_cols + _dim_cols + [_OVERALL_COL]

_rows = []
for _kg in KG_NAMES:
    _row: Dict[str, Optional[float]] = {}
    # Sub-metrics
    for _col_label, _dim, _extractor in SUB_METRICS:
        _ck = ckpts.get(_dim)
        _v  = _extractor(_ck, _kg) if _ck is not None else None
        _row[_col_label] = _v
    # Dim scores
    for _d in DIMS:
        _row[DIM_LABELS.get(_d, _d) + _DIM_SCORE_SUFFIX] = summary_scores[_kg].get(_d)
    # Overall mean across loaded dimensions
    _dim_vals = [summary_scores[_kg].get(_d) for _d in DIMS if _d in summary_scores[_kg]]
    _row[_OVERALL_COL] = round(float(np.mean(_dim_vals)), 4) if _dim_vals else None
    _rows.append(_row)

summary_df = pd.DataFrame(_rows, index=KG_NAMES, columns=_all_cols)
summary_df.index.name = 'KG'

# Export CSV
_csv_out = BASE / 'results' / 'tables' / '00_benchmark_summary.csv'
_csv_out.parent.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(_csv_out, float_format='%.4f')
print(f'CSV exported: {_csv_out}')

# Styled HTML display - values below 0.5 highlighted in red
def _fmt_cell(v):
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return 'N/A'
    return f'{v:.3f}'

def _highlight_low(v):
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return ''
    return f'color: {ALERT_RED}; font-weight: bold' if v < 0.5 else ''

_styled = (
    summary_df.style
    .format(_fmt_cell)
    .map(_highlight_low)
    .set_caption('BioKGSuite - full benchmark summary (values < 0.5 in red)')
    .set_table_styles([
        {'selector': 'caption',
         'props': [('font-size', '11pt'), ('font-weight', 'bold'),
                   ('color', TEXT_COLOR), ('caption-side', 'top')]},
        {'selector': 'th',
         'props': [('font-size', '8pt'), ('text-align', 'center'),
                   ('padding', '4px 6px'), ('white-space', 'nowrap')]},
        {'selector': 'td',
         'props': [('font-size', '8.5pt'), ('text-align', 'center'),
                   ('padding', '3px 6px')]},
    ])
)
_styled

CSV exported: /home/moline/biokgsuite/results/tables/00_benchmark_summary.csv


,Entity Coverage,Relation Coverage,Entity Validity,Relational Consistency,Edge Traceability,Uncertainty Quantification,Connectedness,Small-World,Reachability,Community Purity,Random Dropout,Periph. Dropout,Link Prediction,Neighborhood Retrieval,Multi-hop Reasoning,Data-Sparse Gen.,Cross-Domain Gen.,Prospective Gen.,Coverage [dim],Annotation Accuracy [dim],Trustworthiness [dim],Topology [dim],Stability [dim],Task Performance [dim],Generalisation [dim],Overall Mean
KG,,,,,,,,,,,,,,,,,,,,,,,,,,
primekg,0.777,0.388,1.000,1.000,0.500,0.000,1.000,0.732,0.824,0.544,0.996,0.961,0.946,0.433,0.364,0.691,0.968,0.810,0.583,1.000,0.387,0.775,0.979,0.581,0.823,0.732
hetionet,0.374,0.157,1.000,1.000,0.000,0.000,0.960,0.585,0.826,0.266,0.989,0.990,0.875,0.141,0.190,0.783,0.895,N/A,0.266,1.000,0.247,0.659,0.990,0.402,0.839,0.629
drkg,0.566,0.361,0.998,1.000,1.000,0.000,0.992,0.801,0.907,0.190,0.995,0.932,0.928,0.270,0.350,0.561,0.956,0.695,0.463,0.999,0.476,0.723,0.963,0.516,0.737,0.697
openbilink,0.670,0.173,1.000,1.000,1.000,0.000,1.000,0.928,0.492,0.340,0.935,0.854,0.760,0.537,0.350,0.642,0.838,N/A,0.421,1.000,0.509,0.690,0.894,0.549,0.740,0.686
biokg,0.520,0.410,1.000,1.000,0.000,0.000,0.933,0.857,0.873,0.162,0.999,0.999,0.898,0.147,0.090,0.675,0.890,0.636,0.465,1.000,0.188,0.707,0.999,0.378,0.734,0.639
matrix,0.916,0.634,1.000,1.000,1.000,0.934,0.964,1.000,0.851,0.251,1.000,1.000,0.996,0.323,0.180,0.944,0.998,0.936,0.775,1.000,0.978,0.766,1.000,0.500,0.960,0.854
